# Part 1 - Data Cleaning and Standardisation

## 1. Aim

This notebook cleans and standardises the three raw datasets collected in the previous step:

- text
- images
- audio

The aim is to transform the raw collected materials into consistent and analysable formats for later vectorisation, clustering, and parameter mapping.

This notebook does not yet perform machine learning or 3D generation.  
It focuses only on preparing reliable cleaned datasets.

## 2. Cleaning Strategy

Each dataset is cleaned separately because text, images, and audio require different forms of processing.

### Text cleaning
- keep the original raw text
- create a cleaned text version
- remove noise and standardise formatting

### Image cleaning
- verify file paths
- remove broken images
- standardise image size
- extract basic visual statistics

### Audio cleaning
- verify file paths
- remove broken files
- load audio in a consistent sampling rate
- extract basic acoustic statistics

The cleaned datasets will be saved into the `data/cleaned/` folder.

## 3. Setup

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import os

from PIL import Image
import cv2
import librosa

In [3]:
BASE_DIR = Path.cwd().resolve().parent.parent

RAW_DIR = BASE_DIR / "data" / "raw"
CLEANED_DIR = BASE_DIR / "data" / "cleaned"

TEXT_RAW_DIR = RAW_DIR / "text"
IMAGE_RAW_DIR = RAW_DIR / "images"
AUDIO_RAW_DIR = RAW_DIR / "audio"

TEXT_CLEAN_DIR = CLEANED_DIR / "text"
IMAGE_CLEAN_DIR = CLEANED_DIR / "images"
AUDIO_CLEAN_DIR = CLEANED_DIR / "audio"

TEXT_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("CLEANED_DIR:", CLEANED_DIR)

BASE_DIR: D:\Work\Workspace\Projects\Python\data-driven-surface
RAW_DIR: D:\Work\Workspace\Projects\Python\data-driven-surface\data\raw
CLEANED_DIR: D:\Work\Workspace\Projects\Python\data-driven-surface\data\cleaned


## 4. Load Raw Datasets

In [4]:
text_df = pd.read_csv(TEXT_RAW_DIR / "text_raw.csv")
image_df = pd.read_csv(IMAGE_RAW_DIR / "image_raw.csv")
audio_df = pd.read_csv(AUDIO_RAW_DIR / "audio_raw.csv")

print("Raw text rows:", len(text_df))
print("Raw image rows:", len(image_df))
print("Raw audio rows:", len(audio_df))

Raw text rows: 210
Raw image rows: 231
Raw audio rows: 350


## 5. Text Cleaning

The text dataset contains chunked sections from *Alice’s Adventures in Wonderland*.

The cleaning process keeps the original text while creating a new `clean_text` column for later analysis.

### 5.1 Inspect the raw text data

In [5]:
text_df.head()

,id,chapter,chunk,text
0,alice_ch01_c001,1,1,ALICE'S ADVENTURES IN WONDERLAND Lewis Carroll
1,alice_ch02_c001,2,1,Down the Rabbit-Hole Alice was beginning to ge...
2,alice_ch02_c002,2,2,I shall be late!' (when she thought it over af...
3,alice_ch02_c003,2,3,The rabbit-hole went straight on like a tunnel...
4,alice_ch02_c004,2,4,She took down a jar from one of the shelves as...


### 5.2 Define a text cleaning function

In [6]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)                 # remove HTML-like tags
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)       # remove punctuation/symbols
    text = re.sub(r"\s+", " ", text)                  # normalise whitespace
    
    return text.strip()

### 5.3 Apply text cleaning

In [7]:
# keep raw text explicitly
if "raw_text" not in text_df.columns:
    if "text" in text_df.columns:
        text_df = text_df.rename(columns={"text": "raw_text"})

text_df["clean_text"] = text_df["raw_text"].apply(clean_text)
text_df["length"] = text_df["clean_text"].str.len()
text_df["word_count"] = text_df["clean_text"].str.split().apply(len)

text_df.head()

,id,chapter,chunk,raw_text,clean_text,length,word_count
0,alice_ch01_c001,1,1,ALICE'S ADVENTURES IN WONDERLAND Lewis Carroll,alice s adventures in wonderland lewis carroll,46,7
1,alice_ch02_c001,2,1,Down the Rabbit-Hole Alice was beginning to ge...,down the rabbit hole alice was beginning to ge...,741,148
2,alice_ch02_c002,2,2,I shall be late!' (when she thought it over af...,i shall be late when she thought it over after...,675,134
3,alice_ch02_c003,2,3,The rabbit-hole went straight on like a tunnel...,the rabbit hole went straight on like a tunnel...,640,127
4,alice_ch02_c004,2,4,She took down a jar from one of the shelves as...,she took down a jar from one of the shelves as...,681,141


### 5.4 Remove unusable short text rows

In [8]:
text_clean_df = text_df[text_df["word_count"] >= 10].copy()
text_clean_df = text_clean_df.reset_index(drop=True)

print("Cleaned text rows:", len(text_clean_df))

Cleaned text rows: 209


### 5.5 Save cleaned text data

In [9]:
text_clean_df.to_csv(TEXT_CLEAN_DIR / "text_cleaned.csv", index=False, encoding="utf-8-sig")
print("Saved:", TEXT_CLEAN_DIR / "text_cleaned.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\cleaned\text\text_cleaned.csv


## 6. Image Cleaning

The image dataset contains fantasy and surreal visual references collected from the Pexels API.

The cleaning process checks whether images exist, removes broken files, standardises image size, and extracts basic visual features.

### 6.1 Inspect the raw image data

In [10]:
image_df.head()

,id,source,pexels_id,width,height,photographer,description,image_url,local_path
0,img_0000,pexels,27776534,4000,6000,Ville Aalto,"A unique house atop a rocky cliff with a tree,...",https://images.pexels.com/photos/27776534/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...
1,img_0001,pexels,7391802,4069,2713,Rex Joshua Alarcon America,"Fairytale-inspired castle nestled in Batangas,...",https://images.pexels.com/photos/7391802/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...
2,img_0002,pexels,11592804,2160,2700,Mo Eid,A digital surreal landscape featuring pink clo...,https://images.pexels.com/photos/11592804/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...
3,img_0003,pexels,4835500,3456,5184,raul romagnoli,Creative depiction of a large blue snake along...,https://images.pexels.com/photos/4835500/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...
4,img_0004,pexels,36902049,4032,3024,celal keser,Two whimsical wooden houses nestled in a lush ...,https://images.pexels.com/photos/36902049/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...


### 6.2 Check whether local image files exist

In [11]:
image_df["file_exists"] = image_df["local_path"].apply(lambda x: Path(x).exists() if pd.notna(x) else False)

print(image_df["file_exists"].value_counts())

file_exists
True    231
Name: count, dtype: int64


### 6.3 Keep only valid image files

In [12]:
image_clean_df = image_df[image_df["file_exists"]].copy()
image_clean_df = image_clean_df.reset_index(drop=True)

print("Valid image rows:", len(image_clean_df))

Valid image rows: 231


### 6.4 Standardise image size

In [13]:
def resize_image(path, size=(512, 512)):
    try:
        img = Image.open(path).convert("RGB")
        img = img.resize(size)
        img.save(path)
        return True
    except Exception:
        return False

In [14]:
resize_results = []

for path in image_clean_df["local_path"]:
    resize_results.append(resize_image(path, size=(512, 512)))

image_clean_df["resized"] = resize_results

print(image_clean_df["resized"].value_counts())

resized
True    231
Name: count, dtype: int64


### 6.5 Extract basic visual features

The following features are extracted:
- brightness
- saturation
- edge density

In [15]:
def extract_image_features(path):
    try:
        img = cv2.imread(path)
        if img is None:
            return None
        
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        brightness = np.mean(hsv[:, :, 2])
        saturation = np.mean(hsv[:, :, 1])

        edges = cv2.Canny(img, 100, 200)
        edge_density = edges.mean()

        return brightness, saturation, edge_density
    
    except Exception:
        return None

In [16]:
image_features = image_clean_df["local_path"].apply(extract_image_features)

image_clean_df[["brightness", "saturation", "edge_density"]] = pd.DataFrame(
    image_features.tolist(),
    index=image_clean_df.index
)

image_clean_df.head()

,id,source,pexels_id,width,height,photographer,description,image_url,local_path,file_exists,resized,brightness,saturation,edge_density
0,img_0000,pexels,27776534,4000,6000,Ville Aalto,"A unique house atop a rocky cliff with a tree,...",https://images.pexels.com/photos/27776534/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...,True,True,152.730297,89.531120,31.253414
1,img_0001,pexels,7391802,4069,2713,Rex Joshua Alarcon America,"Fairytale-inspired castle nestled in Batangas,...",https://images.pexels.com/photos/7391802/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...,True,True,114.278633,105.534653,9.059200
2,img_0002,pexels,11592804,2160,2700,Mo Eid,A digital surreal landscape featuring pink clo...,https://images.pexels.com/photos/11592804/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...,True,True,149.746407,69.439190,16.839237
3,img_0003,pexels,4835500,3456,5184,raul romagnoli,Creative depiction of a large blue snake along...,https://images.pexels.com/photos/4835500/pexel...,D:\Work\Workspace\Projects\Python\data-driven-...,True,True,126.662529,79.775913,19.733162
4,img_0004,pexels,36902049,4032,3024,celal keser,Two whimsical wooden houses nestled in a lush ...,https://images.pexels.com/photos/36902049/pexe...,D:\Work\Workspace\Projects\Python\data-driven-...,True,True,80.581467,78.962467,55.836697


### 6.6 Save cleaned image data

In [17]:
image_clean_df.to_csv(IMAGE_CLEAN_DIR / "image_cleaned.csv", index=False, encoding="utf-8-sig")
print("Saved:", IMAGE_CLEAN_DIR / "image_cleaned.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\cleaned\images\image_cleaned.csv


## 7. Audio Cleaning

The audio dataset contains non-linguistic sound previews collected from the Freesound API.

The cleaning process verifies file paths, removes broken files, and extracts basic audio features.

### 7.1 Inspect the raw audio data

In [18]:
audio_df.head()

,id,source,freesound_id,name,username,license,duration,tags,description,audio_url,local_path
0,audio_0000,freesound,383184,chattering teeth - winding up,ChrisReierson,https://creativecommons.org/licenses/by/4.0/,9.77104,"chattering, effect, funny, humor, mechanical, ...",Winding up a miniature chattering teeth toy. I...,https://cdn.freesound.org/previews/383/383184_...,D:\Work\Workspace\Projects\Python\data-driven-...
1,audio_0001,freesound,383169,wind gap indoors 002 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,64.16610,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383169_...,D:\Work\Workspace\Projects\Python\data-driven-...
2,audio_0002,freesound,383139,wind gap indoors 001 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,110.95900,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383139_...,D:\Work\Workspace\Projects\Python\data-driven-...
3,audio_0003,freesound,379470,Wind3.mp3,vandale,http://creativecommons.org/publicdomain/zero/1.0/,5.01406,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379470_...,D:\Work\Workspace\Projects\Python\data-driven-...
4,audio_0004,freesound,379467,Wind2.ogg,vandale,http://creativecommons.org/publicdomain/zero/1.0/,3.95977,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379467_...,D:\Work\Workspace\Projects\Python\data-driven-...


### 7.2 Check whether local audio files exist

In [19]:
audio_df["file_exists"] = audio_df["local_path"].apply(lambda x: Path(x).exists() if pd.notna(x) else False)

print(audio_df["file_exists"].value_counts())

file_exists
True    350
Name: count, dtype: int64


### 7.3 Keep only valid audio files

In [20]:
audio_clean_df = audio_df[audio_df["file_exists"]].copy()
audio_clean_df = audio_clean_df.reset_index(drop=True)

print("Valid audio rows:", len(audio_clean_df))

Valid audio rows: 350


### 7.4 Extract basic audio features

The following features are extracted:
- duration
- RMS energy
- zero-crossing rate
- spectral centroid

In [21]:
def extract_audio_features(path):
    try:
        y, sr = librosa.load(path, sr=22050, mono=True)
        
        duration = librosa.get_duration(y=y, sr=sr)
        rms = np.mean(librosa.feature.rms(y=y))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))

        return duration, sr, rms, zcr, spectral_centroid
    
    except Exception:
        return None

In [23]:
audio_features = audio_clean_df["local_path"].apply(extract_audio_features)

audio_clean_df[["duration_clean", "sr", "rms", "zcr", "spectral_centroid"]] = pd.DataFrame(
    audio_features.tolist(),
    index=audio_clean_df.index
)

audio_clean_df.head()

,id,source,freesound_id,name,username,license,duration,tags,description,audio_url,local_path,file_exists,duration_clean,sr,rms,zcr,spectral_centroid
0,audio_0000,freesound,383184,chattering teeth - winding up,ChrisReierson,https://creativecommons.org/licenses/by/4.0/,9.77104,"chattering, effect, funny, humor, mechanical, ...",Winding up a miniature chattering teeth toy. I...,https://cdn.freesound.org/previews/383/383184_...,D:\Work\Workspace\Projects\Python\data-driven-...,True,9.771066,22050,0.007996,0.212267,4631.344945
1,audio_0001,freesound,383169,wind gap indoors 002 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,64.16610,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383169_...,D:\Work\Workspace\Projects\Python\data-driven-...,True,64.166168,22050,0.075333,0.012516,457.364479
2,audio_0002,freesound,383139,wind gap indoors 001 170305_1100.wav,klankbeeld,https://creativecommons.org/licenses/by/4.0/,110.95900,"air-flow, air-stream, ajar, ambience, ambient,...",The wind ajar trough a door-gap indoors (10cm ...,https://cdn.freesound.org/previews/383/383139_...,D:\Work\Workspace\Projects\Python\data-driven-...,True,110.958549,22050,0.065641,0.009511,430.598940
3,audio_0003,freesound,379470,Wind3.mp3,vandale,http://creativecommons.org/publicdomain/zero/1.0/,5.01406,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379470_...,D:\Work\Workspace\Projects\Python\data-driven-...,True,5.014059,22050,0.056515,0.071999,1899.045851
4,audio_0004,freesound,379467,Wind2.ogg,vandale,http://creativecommons.org/publicdomain/zero/1.0/,3.95977,"air, breeze, gust, swoosh, wind",wind breeze swoosh air gust,https://cdn.freesound.org/previews/379/379467_...,D:\Work\Workspace\Projects\Python\data-driven-...,True,3.959773,22050,0.024671,0.060981,1473.670996


### 7.5 Save cleaned audio data

In [24]:
audio_clean_df.to_csv(AUDIO_CLEAN_DIR / "audio_cleaned.csv", index=False, encoding="utf-8-sig")
print("Saved:", AUDIO_CLEAN_DIR / "audio_cleaned.csv")

Saved: D:\Work\Workspace\Projects\Python\data-driven-surface\data\cleaned\audio\audio_cleaned.csv


## 8. Cleaning Summary

At the end of this notebook:

- the text dataset has been cleaned and standardised
- the image dataset has been validated and enriched with basic visual features
- the audio dataset has been validated and enriched with basic acoustic features

These cleaned datasets are now ready for vectorisation and comparison in the next notebook.

In [25]:
summary = {
    "text_raw_rows": len(text_df),
    "text_clean_rows": len(text_clean_df),
    "image_raw_rows": len(image_df),
    "image_clean_rows": len(image_clean_df),
    "audio_raw_rows": len(audio_df),
    "audio_clean_rows": len(audio_clean_df),
}

summary_df = pd.DataFrame([summary])
summary_df

,text_raw_rows,text_clean_rows,image_raw_rows,image_clean_rows,audio_raw_rows,audio_clean_rows
0,210,209,231,231,350,350


## Next Step

The next notebook will focus on vectorisation and method comparison.

This includes:

- text: TF-IDF vs embeddings
- image: handcrafted features vs CLIP/CNN features
- audio: spectral features vs MFCC